### This notebook combines all the raw measured burst properties from different telescopes and measuring techniques and massages them in the final burst properties table as presented in the paper.

##### Some code is very hardcoded, but you will have to live with that.

In [1]:
import numpy as np
import pandas as pd
from tqdm import tqdm
from astropy import units as u
pd.set_option("display.max_rows", 80)

In [2]:
def bandwidth_calc(df, telescope, id):
    '''
    input: pandas dataframe where each line has the burst properties per component
    return the bandwith of a burst
    '''
    obs_band = df.f0_MHz.values[0]
    
    #Pband
    if obs_band == 328.0:
        bw_obs = 54
        
    #Lband
    elif obs_band:
        bw_obs = 128

    if telescope == 'st':
        bw_obs = 98

    if id == "B110-tr" or id == "B122-tr":
        bw_obs= 256
    #We take the absolute value, since the fwidth values can be negative
    #The bandwidth comes from autocorrelation and we define everything larger than FWHM as the BW
    signal_bw = int(abs(df.fwidth_sigma_MHz.values.max()) * 2.355)
    if signal_bw > bw_obs * 0.75:
        signal_bw = bw_obs

    return signal_bw, obs_band     

def toa_calulator(df):
    '''
    Return the toa of a burst
    in the case of multiple components of a single burst, take the middle of the burst
    '''
    if df.shape[0] == 1:
        mjd = df.iloc[0]["toa_bary_tdb_inf_freq"]

    elif df.shape[0] > 1:
        mjd_list = df["toa_bary_tdb_inf_freq"].values
        mjd = (mjd_list[0] + mjd_list[-1]) / 2
 
    return mjd

def toa_utc_calulator(df):
    '''
    Return the toa of a burst
    in the case of multiple components of a single burst, take the middle of the burst
    '''
    if df.shape[0] == 1:
        mjd = df.iloc[0]["toa_mjd_utc"]

    elif df.shape[0] > 1:
        mjd_list = df["toa_mjd_utc"].values
        mjd = (mjd_list[0] + mjd_list[-1]) / 2
 
    return mjd
    
def peak_sn_calc(df):
    '''
    input: pandas dataframe where each line has the burst properties per component
    Return the maximum value of the snr of all comp
    '''
    num_comp = df.shape[0]
    if num_comp == 0:
        print(f"WRONG WRONG : {df}")
    if num_comp == 1:
        peak_sn = df.iloc[0]["peak_snr"]
    elif num_comp > 1:
        peak_list = df["peak_snr"].values
        peak_sn = np.max(peak_list)
    
    return peak_sn,num_comp

def peak_sn_calc(df):
    '''
    input: pandas dataframe where each line has the burst properties per component
    Return the maximum value of the snr of all comp
    '''
    num_comp = df.shape[0]
    if num_comp == 0:
        print(f"WRONG WRONG : {df}")
    if num_comp == 1:
        peak_sn = df.iloc[0]["peak_snr"]
    elif num_comp > 1:
        peak_list = df["peak_snr"].values
        peak_sn = np.max(peak_list)
    
    return peak_sn,num_comp

def isotropic_energies(fluences, distance=616):
    "Convert list of fluences to spectral densities"
    "Method based on fluence.py by K.Nimmo"

    #convert from jyms to jys
    fluence_jys = np.array(fluences) * 1e-3

    dis = 1.0 * u.megaparsec
    megap_cm = dis.to(u.cm).value

    #mpc to cm
    distance_lum_cm = megap_cm*distance

    #redshift correction, see evernote for details
    #redshift is from: Snelders et al
    #distance is also from Ravi et al, FAST converts themselves based on PLANCK 2016
    #link: https://ui.adsabs.harvard.edu/abs/2025ApJ...992L..35B/abstract
    z = 0.130287
    red_cor = 1 / (1 + z)**2

    energy_iso = fluence_jys * 4*np.pi*(distance_lum_cm**2) * 1e-23 * red_cor
    #print(energy_iso)

    return energy_iso

def speclum(fluence, ontime, distance=616, z=0.130287):
    '''
    fluence assumed in Jy~ms
    ontime assumed in ms
    distance assumed in Mpc
    '''
    # actual ontime or total width? Width gives a lover limit on the luminosity.
    
    #convert Jy ms to J s; and ms to s
    fluence_jys = fluence*1e-3
    ontime /= 1e3
    
    #convert Mpc to cm
    distance_lum_cm = 3.086e24*distance
    energy_iso= fluence_jys*4*np.pi*(distance_lum_cm**2)*1e-23 / (1+z)**(2)
    lum_spec = energy_iso/ontime
    
    return lum_spec

In [3]:
def fluence_looper(df_fluence, two_bit_flag=False):
    """
    function to sum over the fluences components of bursts
    
    Input: The burst_csv file with all information, spc/sfxc/scale
    This is needed because we return a df with fluence (spc) and toa (sfxc)
    
    return: pandas dataframe with: id/fluence/telescope/c_freq/toa/width_ms
    """

    #the burst names to loop over
    exps = df_fluence['id'].unique()

    #Create a list with the telescope names for easy filtering later on
    exps_tel = [i.split('-')[1] for i in exps]

    #loop over every burst to sum the fluence
    info_list = []
    for exp, telescope in zip(exps, exps_tel):
        
        #Fluence: Making a small temp df for each exp, only taking the spc values
        if telescope != 'st' and two_bit_flag == False:

            ## This coding line reads like garbage - but deal with it because it works
            if exp == "B125-o8" or exp == "B157-tr":
                fluence_df_exp_temp = df_fluence[df_fluence['id'] == exp]
            
                #TOA
                toa = toa_calulator(fluence_df_exp_temp) 
                toa_utc = toa_utc_calulator(fluence_df_exp_temp) 
                
            else:
                fluence_df_exp_temp = df_fluence[(df_fluence['id'] == exp) & (df_fluence['src'] == 'spc')]
                
                #TOA
                fluence_df_exp_temp_mjd = df_fluence[(df_fluence['id'] == exp) & (df_fluence['src'] == 'sfxc')]
                toa = toa_calulator(fluence_df_exp_temp_mjd) 
                toa_utc = toa_utc_calulator(fluence_df_exp_temp_mjd)
                
        elif telescope == 'st' or two_bit_flag==True:
            fluence_df_exp_temp = df_fluence[df_fluence['id'] == exp]
            
            #TOA
            toa = toa_calulator(fluence_df_exp_temp) 
            toa_utc = toa_utc_calulator(fluence_df_exp_temp) 

        #Calculate the total fluence
        fluence = sum(fluence_df_exp_temp['fluence_jyms'].values)

        #Calculate isotropic energies
        energies_isotropic = isotropic_energies(fluence)
        
        #Calculate the max SN of a burst
        peak_sn, num_comp = peak_sn_calc(fluence_df_exp_temp)
    
        #Calculate the bandwith
        signal_bw, cent_freq = bandwidth_calc(fluence_df_exp_temp, telescope, id=exp)
        
        #width in ms
        width_t = fluence_df_exp_temp.end_acf_range.values.max() - fluence_df_exp_temp.begin_acf_range.values.min()

        #Calculate the spectral luminosity
        spel_lum = speclum(fluence, width_t, distance=616, z=0.130287)
        
        #Since the F0 is the same for all components, take all components, take the first entry
        cent_freq = fluence_df_exp_temp['f0_MHz'].values[0]
        
        #Add to the list
        info_list.append([toa, peak_sn, fluence, num_comp, width_t, energies_isotropic, spel_lum, \
                          signal_bw, cent_freq, two_bit_flag, toa_utc])                          
        
    #Convert the fluence list to array
    info_list_arr = np.asarray(info_list)
    
    #Convert the info to dataframe
    new_data = {'id': exps, 'station': exps_tel, 'toa': info_list_arr[:,0], \
                'peak_sn': info_list_arr[:,1], 'fluence': info_list_arr[:,2], \
                'width_ms': info_list_arr[:,4], \
                'bandwidth': info_list_arr[:,7], \
                'central frequency': info_list_arr[:,8], \
                'spectral-density': info_list_arr[:,5],\
                'spectral-luminosity': info_list_arr[:,6], \
                '2bit-wb-tag': info_list_arr[:,9],
                'toa_utc': info_list_arr[:,10]}
    
    df_new_fluence = pd.DataFrame(data=new_data)

    #add only the index number as a seperate column for potential later use
    df_new_fluence['burst-index'] = df_new_fluence['id'].str.split('-').str[0].str[1:].astype(int)
    
    return df_new_fluence

In [4]:
def df_to_small_table(file):
    """
    Function to load in the burst.csv file and return the shortend version of the .csv file
    """

    #load in the main csv file
    fluence_df = pd.read_csv(file, index_col=0, header=0, na_values='NA', engine='python').reset_index(drop=True)
    fluence_df.sort_values(by=['id'])

    if file == "2bit_westerbork_bursts_r147.csv":
        two_bit_flag = True
    else:
        two_bit_flag = False

    #B86-88-tr were to faint to quantify burst properties
    fluence_df = fluence_df[~fluence_df['id'].isin(['B86-tr', 'B88-tr'])]
    
    df_combined = fluence_looper(fluence_df, two_bit_flag)    
    df_combined.sort_values(by=['id'])

    return df_combined

def df_formatter(df_combined):
    
    #Resort and reset the df
    df_combined = df_combined.sort_values(by=['toa'])
    df_combined = df_combined.reset_index(drop=True)
    
    #Setting the formatting of the numbers correct
    df_combined["bandwidth"] = df_combined["bandwidth"].astype(int)
    df_combined["central frequency"] = df_combined["central frequency"].astype(int)
    
    df_combined['fluence_err'] = pd.to_numeric(df_combined['fluence'] * 0.2, errors='coerce').round(2)
    
    # #Minimum spectral energy is on the order of 1e30 // calculated with the .min()
    df_combined['spectral-density'] /= 1e30
    df_combined['spec_den_err'] = df_combined['spectral-density'] * 0.2
    df_combined['spectral-density'] = df_combined['spectral-density'].round(2)
    df_combined['spec_den_err'] = df_combined['spec_den_err'].round(2)
    
    # #Minimum spectral lum is on the order of 1e31 // calculated with the .min()
    df_combined['spectral-luminosity'] /= 1e32
    df_combined['spec_lum_err'] = df_combined['spectral-luminosity'] * 0.2
    df_combined['spectral-luminosity'] = df_combined['spectral-luminosity'].round(2)
    df_combined['spec_lum_err'] = df_combined['spec_lum_err'].round(2)

    df_combined['peak_sn'] = pd.to_numeric(df_combined['peak_sn'], errors='coerce').round(2)
    df_combined['width_ms'] = pd.to_numeric(df_combined['width_ms'], errors='coerce').round(2)
    df_combined['fluence'] = pd.to_numeric(df_combined['fluence'], errors='coerce').round(2)
    
    df_combined['peak_sn'] = df_combined['peak_sn'].round(2)
    df_combined['width_ms'] = df_combined['width_ms'].round(2)
    df_combined['fluence'] = df_combined['fluence'].round(2)

    df_combined["2bit-wb-tag"] = df_combined["2bit-wb-tag"].astype(int)
    
    df_paper = df_combined[['id', 'station', 'toa', 'toa_utc', 'peak_sn', 'fluence', 'fluence_err',\
                            'width_ms', 'spectral-density', 'spec_den_err', 'spectral-luminosity', 'spec_lum_err',\
                            'bandwidth', 'central frequency', '2bit-wb-tag']]

    new_rows = pd.DataFrame([
    {
        "id": "B48-wb",
        "station": "wb",
    },
    {
        "id": "B81-o8",
        "station": "o8",
    },
    {
        "id": "B86-tr",
        "station": "tr",
    },
    {
        "id": "B88-tr",
        "station": "tr",
    }
    ])
    df_paper = pd.concat([df_paper, new_rows], ignore_index=True)

    df_combined = df_combined.sort_values(by=['toa'])

    return df_paper

In [5]:
def df_combiner():

    df_westerbork_2bit = df_to_small_table(file="2bit_westerbork_bursts_r147.csv")
    df_stockert = df_to_small_table(file="stockert_bursts_r147.csv")
    df_westerbork_pband = df_to_small_table(file="westerbork_pband.csv")
    df_onsala = df_to_small_table(file="hyperflash_onsala_L.csv")
    df_torun = df_to_small_table(file="hyperflash_torun_L.csv")
    df_westerbork = df_to_small_table(file="hyperflash_westerbork_L.csv")
    
    dfs = [
        df_westerbork_2bit,
        df_stockert,
        df_westerbork_pband,
        df_onsala,
        df_torun,
        df_westerbork,
    ]
    
    df_all = pd.concat(dfs, ignore_index=True)
    df_all = df_all.sort_values(by=['toa'])
    #display(df_all)

    df_paper = df_formatter(df_all)
    display(df_paper)

    name_csv = "FRB20240114A_HyperFlash_paper_table_v2.csv"
    df_paper.to_csv(name_csv, header=True, index=True, na_rep='NA')
    
    return df_paper
    
df_paper = df_combiner()

,id,station,toa,toa_utc,peak_sn,fluence,fluence_err,width_ms,spectral-density,spec_den_err,spectral-luminosity,spec_lum_err,bandwidth,central frequency,2bit-wb-tag
0,B01-wb,wb,60341.533745,60341.538213,8.08,203.23,40.65,45.06,72.22,14.44,16.03,3.21,40.0,328.0,0.0
1,B02-wb,wb,60355.639527,60355.644167,5.26,16.66,3.33,14.34,5.92,1.18,4.13,0.83,128.0,1271.0,0.0
2,B03-st,st,60357.339134,60357.343782,8.60,20.15,4.03,3.71,7.16,1.43,19.28,3.86,98.0,1381.0,0.0
3,B04-st,st,60357.433876,60357.438523,6.00,25.10,5.02,9.61,8.92,1.78,9.28,1.86,53.0,1381.0,0.0
4,B05-st,st,60361.542887,60361.547509,5.21,35.49,7.10,7.86,12.61,2.52,16.04,3.21,98.0,1381.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
211,B180-tr,tr,60819.332590,60819.331089,7.58,7.22,1.44,7.17,2.57,0.51,3.58,0.72,95.0,1444.0,0.0
212,B48-wb,wb,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
213,B81-o8,o8,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
214,B86-tr,tr,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [13]:
df_2bit = df_paper[df_paper["2bit-wb-tag"] == 1.0]
df_2bit = df_2bit.sort_values(by=['fluence'])
display(df_2bit)
print(len(df_2bit.index))

,id,station,toa,toa_utc,peak_sn,fluence,fluence_err,width_ms,spectral-density,spec_den_err,spectral-luminosity,spec_lum_err,bandwidth,central frequency,2bit-wb-tag
203,B172-wb,wb,60807.267393,60807.267029,5.43,10.85,2.17,10.24,3.86,0.77,3.77,0.75,71.0,1271.0,1.0
16,B17-wb,wb,60373.253667,60373.258074,6.47,11.49,2.30,4.10,4.08,0.82,9.97,1.99,128.0,1271.0,1.0
52,B52-wb,wb,60379.265983,60379.270192,5.37,12.10,2.42,6.14,4.30,0.86,7.00,1.40,128.0,1271.0,1.0
190,B161-wb,wb,60696.570776,60696.574916,4.65,13.77,2.75,11.26,4.89,0.98,4.35,0.87,128.0,1271.0,1.0
31,B31-wb,wb,60375.426785,60375.431127,6.75,16.15,3.23,13.31,5.74,1.15,4.31,0.86,128.0,1271.0,1.0
9,B10-wb,wb,60370.262306,60370.266790,5.10,16.21,3.24,11.26,5.76,1.15,5.11,1.02,128.0,1271.0,1.0
178,B150-wb,wb,60682.494586,60682.498021,6.36,17.23,3.45,10.24,6.12,1.22,5.98,1.20,128.0,1271.0,1.0
26,B27-wb,wb,60375.380475,60375.384819,6.38,17.39,3.48,7.17,6.18,1.24,8.63,1.73,128.0,1271.0,1.0
6,B07-wb,wb,60369.290731,60369.295237,5.47,17.45,3.49,4.86,6.20,1.24,12.75,2.55,128.0,1271.0,1.0
201,B170-wb,wb,60805.359486,60805.359298,15.12,17.65,3.53,2.82,6.27,1.25,22.28,4.46,96.0,1271.0,1.0


20


In [14]:
df_wb_non2bit = df_paper[df_paper["station"] == "wb"]
df_wb_non2bit = df_wb_non2bit.sort_values(by=['fluence'])
display(df_wb_non2bit)
print(len(df_wb_non2bit.index))

,id,station,toa,toa_utc,peak_sn,fluence,fluence_err,width_ms,spectral-density,spec_den_err,spectral-luminosity,spec_lum_err,bandwidth,central frequency,2bit-wb-tag
146,B123-wb,wb,60424.432518,60424.433747,5.44,6.09,1.22,3.07,2.16,0.43,7.05,1.41,78.0,1271.0,0.0
8,B09-wb,wb,60369.366278,60369.370769,4.43,7.34,1.47,22.53,2.61,0.52,1.16,0.23,77.0,1271.0,0.0
10,B11-wb,wb,60370.289589,60370.294059,4.85,8.67,1.73,11.26,3.08,0.62,2.74,0.55,128.0,1271.0,0.0
184,B156-wb,wb,60683.682305,60683.685796,4.94,9.48,1.90,3.58,3.37,0.67,9.40,1.88,128.0,1271.0,0.0
133,B110-wb,wb,60387.363333,60387.367176,5.88,10.14,2.03,2.56,3.61,0.72,14.09,2.82,128.0,1271.0,0.0
33,B32-wb,wb,60375.444173,60375.448500,4.54,10.33,2.07,12.29,3.67,0.73,2.99,0.60,128.0,1271.0,0.0
203,B172-wb,wb,60807.267393,60807.267029,5.43,10.85,2.17,10.24,3.86,0.77,3.77,0.75,71.0,1271.0,1.0
13,B14-wb,wb,60370.453285,60370.457751,6.20,11.13,2.23,3.33,3.95,0.79,11.88,2.38,128.0,1271.0,0.0
16,B17-wb,wb,60373.253667,60373.258074,6.47,11.49,2.30,4.10,4.08,0.82,9.97,1.99,128.0,1271.0,1.0
52,B52-wb,wb,60379.265983,60379.270192,5.37,12.10,2.42,6.14,4.30,0.86,7.00,1.40,128.0,1271.0,1.0


62


In [20]:
df_paper_table = pd.concat([df_paper.head(6), df_paper.tail(10)])
display(df_paper_table)

,id,station,toa,toa_utc,peak_sn,fluence,fluence_err,width_ms,spectral-density,spec_den_err,spectral-luminosity,spec_lum_err,bandwidth,central frequency,2bit-wb-tag
0,B01-wb,wb,60341.533745,60341.538213,8.08,203.23,40.65,45.06,72.22,14.44,16.03,3.21,40.0,328.0,0.0
1,B02-wb,wb,60355.639527,60355.644167,5.26,16.66,3.33,14.34,5.92,1.18,4.13,0.83,128.0,1271.0,0.0
2,B03-st,st,60357.339134,60357.343782,8.60,20.15,4.03,3.71,7.16,1.43,19.28,3.86,98.0,1381.0,0.0
3,B04-st,st,60357.433876,60357.438523,6.00,25.10,5.02,9.61,8.92,1.78,9.28,1.86,53.0,1381.0,0.0
4,B05-st,st,60361.542887,60361.547509,5.21,35.49,7.10,7.86,12.61,2.52,16.04,3.21,98.0,1381.0,0.0
5,B06-st,st,60366.341592,60366.346149,5.26,16.72,3.34,3.50,5.94,1.19,17.00,3.40,98.0,1381.0,0.0
206,B175-wb,wb,60814.157155,60814.156148,9.01,27.84,5.57,12.29,9.90,1.98,8.05,1.61,128.0,1271.0,1.0
207,B176-tr,tr,60816.177709,60816.176500,4.85,7.27,1.45,7.17,2.58,0.52,3.61,0.72,128.0,1444.0,0.0
208,B177-wb,wb,60816.333009,60816.331785,4.82,23.81,4.76,10.75,8.46,1.69,7.87,1.57,128.0,1271.0,0.0
209,B179-wb,wb,60817.072561,60817.071283,8.03,21.07,4.21,12.29,7.49,1.50,6.09,1.22,128.0,1271.0,1.0


In [21]:
print(df_paper['id'].values)

['B01-wb' 'B02-wb' 'B03-st' 'B04-st' 'B05-st' 'B06-st' 'B07-wb' 'B08-st'
 'B09-wb' 'B10-wb' 'B11-wb' 'B12-wb' 'B13-wb' 'B14-wb' 'B15-st' 'B16-st'
 'B17-wb' 'B18-wb' 'B19-wb' 'B20-st' 'B21-st' 'B22-st' 'B23-st' 'B24-st'
 'B25-wb' 'B26-wb' 'B27-wb' 'B28-wb' 'B29-wb' 'B30-st' 'B30-wb' 'B31-wb'
 'B31-st' 'B32-wb' 'B33-wb' 'B34-st' 'B35-st' 'B36-st' 'B37-st' 'B38-st'
 'B39-st' 'B40-st' 'B41-st' 'B42-st' 'B43-st' 'B44-st' 'B45-st' 'B46-wb'
 'B47-wb' 'B49-st' 'B50-st' 'B51-st' 'B52-wb' 'B53-st' 'B54-st' 'B55-st'
 'B56-st' 'B56-o8' 'B57-st' 'B57-o8' 'B58-st' 'B58-tr' 'B59-st' 'B60-st'
 'B60-tr' 'B61-st' 'B61-tr' 'B62-st' 'B63-o8' 'B64-tr' 'B65-o8' 'B65-tr'
 'B66-st' 'B67-o8' 'B66-tr' 'B68-o8' 'B68-tr' 'B70-o8' 'B71-o8' 'B72-o8'
 'B73-st' 'B73-o8' 'B74-st' 'B74-o8' 'B75-st' 'B75-o8' 'B76-st' 'B76-o8'
 'B77-o8' 'B78-o8' 'B79-o8' 'B80-st' 'B80-o8' 'B82-o8' 'B83-o8' 'B84-o8'
 'B85-o8' 'B85-tr' 'B86-o8' 'B87-st' 'B87-o8' 'B87-tr' 'B88-st' 'B88-o8'
 'B89-o8' 'B90-o8' 'B90-tr' 'B91-o8' 'B92-o8' 'B93-